# Academic calendar and public safety
## A4 · Fall 2024 event-day comparisons

How many alcohol-related collisions did BPD record on home football, move-in, Halloween, and finals dates compared with ordinary days? We count a collision when its detailed primary collision factor explicitly mentions alcohol or a blood alcohol concentration threshold (`alcohol_pcf_status == "explicit_alcohol"`).

For each event day, average the counts on eligible dates in the same year and month that fall on the same weekday. A control date can serve multiple events.

**Inputs**  
`data/calendar/daily_calendar.csv` and `data/processed/collisions.csv`

**Output**  
`data/processed/a4_collision_matches.csv`

In [1]:
from pathlib import Path

import pandas as pd

# Find the data directory when running from the repository root or notebooks folder.
root = Path.cwd()
if root.name == "notebooks":
    root = root.parent
inputs = root / "data"
# Require one row per local date so calendar lookups identify a single day.
calendar = pd.read_csv(inputs / "calendar/daily_calendar.csv", parse_dates=["date"])
if not calendar["date"].is_unique:
    raise ValueError("Expected one calendar row per date")
calendar = calendar.set_index("date").sort_index()
# Use the local collision date to match the Berkeley calendar.
collisions = pd.read_csv(inputs / "processed/collisions.csv", usecols=["Date", "alcohol_pcf_status"])
collisions["Date"] = pd.to_datetime(collisions["Date"], format="%Y-%m-%d")
print(f"Snapshot records without a local date: {collisions['Date'].isna().sum()}")

Snapshot records without a local date: 101


In [2]:
events = ["home_football", "move_in", "halloween", "finals"]
excluded = events + ["holiday", "non_instructional", "rrr", "instruction_begins",
                     "formal_classes_end", "last_instruction_day", "winter_break"]
# A semester day can serve as a control if all named event and exclusion flags are zero.
ordinary = calendar.index[calendar[excluded].eq(0).all(axis=1)]

# Zero here means zero dated records flagged in this snapshot on that day.
flagged = collisions.loc[collisions["alcohol_pcf_status"].eq("explicit_alcohol")]
counts = flagged.groupby("Date").size().reindex(calendar.index, fill_value=0)

# Preserve unmatched event dates with an empty control pool and missing control mean.
records = []
for event in events:
    for day in calendar.index[calendar[event].eq(1)]:
        # Match within the same year and month on weekday to account for weekly and seasonal patterns.
        controls = ordinary[(ordinary.to_period("M") == day.to_period("M"))
                            & (ordinary.day_of_week == day.weekday())]
        records.append({
            "event": event,
            "event_date": day.date().isoformat(),
            "control_dates": ", ".join(controls.strftime("%Y-%m-%d")),
            "n_controls": len(controls),
            "event_alcohol_collisions": counts.at[day],
            "mean_control_alcohol_collisions": counts.loc[controls].mean(),
        })

matches = pd.DataFrame(records)
output = root / "data/processed/a4_collision_matches.csv"
matches.to_csv(output, index=False)
matches

,event,event_date,control_dates,n_controls,event_alcohol_collisions,mean_control_alcohol_collisions
0,home_football,2024-08-31,2024-08-24,1,0,0.000000
1,home_football,2024-09-14,"2024-09-07, 2024-09-21, 2024-09-28",3,0,0.333333
2,home_football,2024-10-05,2024-10-12,1,0,0.000000
3,home_football,2024-10-19,2024-10-12,1,0,0.000000
4,home_football,2024-10-26,2024-10-12,1,0,0.000000
5,home_football,2024-11-16,"2024-11-02, 2024-11-09, 2024-11-30",3,1,0.000000
6,home_football,2024-11-23,"2024-11-02, 2024-11-09, 2024-11-30",3,1,0.000000
7,move_in,2024-08-20,2024-08-27,1,0,0.000000
8,move_in,2024-08-21,,0,0,NaN
9,halloween,2024-10-31,"2024-10-03, 2024-10-10, 2024-10-17, 2024-10-24",4,0,0.000000


In [3]:
# Each matched event date receives equal weight, regardless of its pool size.
matched = matches.loc[matches["n_controls"].gt(0)]
summary = matches.groupby("event").agg(
    event_days=("event_date", "size"),
    all_event_alcohol_collisions=("event_alcohol_collisions", "sum"),
)
summary.join(matched.groupby("event").agg(
    matched_days=("event_date", "size"),
    mean_event_alcohol_collisions=("event_alcohol_collisions", "mean"),
    mean_control_alcohol_collisions=("mean_control_alcohol_collisions", "mean"),
))

,event_days,all_event_alcohol_collisions,matched_days,mean_event_alcohol_collisions,mean_control_alcohol_collisions
event,,,,,
finals,5,0,4,0.000000,0.000000
halloween,1,0,1,0.000000,0.000000
home_football,7,2,7,0.285714,0.047619
move_in,2,0,1,0.000000,0.000000


### Interpretation

All seven home games had eligible comparisons. Two flagged collisions occurred on game days. Move-in on August 21 and finals on December 20 had no eligible comparison date; their control means are missing. The sparse counts describe recorded primary factors and support only a descriptive comparison for this semester. Noise and assault comparisons require the shared inputs and their coverage checks.